In [45]:
from smolagents import EMPTY_PROMPT_TEMPLATES

system_prompt = """
You are a STRICT ReAct SQL agent.
You will be given a task to solve as best you can.
To do so, you have been given access to the tool 'sql_engine': this tool is basically a Python function which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.


You MUST follow this loop EXACTLY:

Thought
→ sql_engine
→ reflect_on_observation
→ Thought
→ sql_engine
→ reflect_on_observation
→ ...
→ Final Answer

RULES (ABSOLUTE):

1. You are FORBIDDEN from calling sql_engine twice in a row.
2. Every sql_engine call MUST be followed by reflect_on_observation.
3. In reflect_on_observation you MUST:
   - Explain what row_count means
   - Explain whether the query logic is correct
   - Explain what you will do next
4. You are FORBIDDEN from producing the Final Answer
   unless the previous step was reflect_on_observation.
5. The Final Answer MUST be ONLY a SQL query string.

If you skip reflection, the answer is invalid.

"""


my_templates_dict = EMPTY_PROMPT_TEMPLATES.copy()
my_templates_dict["system_prompt"] = system_prompt

In [ ]:
import os
from sqlalchemy import create_engine, inspect, text
from smolagents import ToolCallingAgent, tool
from smolagents.models import InferenceClientModel

# SETUP DATABASE 
db_path = "sqlite:///shakespeare.sqlite"
if not os.path.exists("shakespeare.sqlite"):
    print("WARNING: 'shakespeare.sqlite' not found. Please upload it.")

engine = create_engine(db_path)
inspector = inspect(engine)

# GENERATE SCHEMA CONTEXT 
table_names = inspector.get_table_names()
schema_text = "## Database Schema\n"
for table in table_names:
    schema_text += f"Table: {table}\n"
    columns = inspector.get_columns(table)
    for col in columns:
        schema_text += f"  - {col['name']} ({col['type']})\n"

# TOOLS 
@tool
def sql_engine(reasoning: str, query: str) -> dict:
    """
    Executes a SQL query to validate logic.

    Args:
        reasoning (str): Why this query is executed.
        query (str): SQL query to execute.

    Returns:
        dict: Structured observation for reasoning.
    """
    print(f"\n[THOUGHT]: {reasoning}")

    try:
        with engine.connect() as con:
            result = con.execute(text(query))
            rows = [dict(row._mapping) for row in result]

        observation = {
            "row_count": len(rows),
            "sample_rows": rows[:5]
        }

        #print(f"[OBSERVATION]: {observation}")
        return observation

    except Exception as e:
        return {"error": str(e)}
    
@tool
def reflect_on_observation(thought: str, observation: dict) -> str:
    """
    Forces the agent to explain what the observation means and what to do next.

    Args:
        thought (str): Explanation of how the observation affects the next step.
        observation (dict): Output returned by sql_engine.

    Returns:
        str: Reflection confirmation.
    """
    #print("\n[OBSERVATION]:", observation)
    #print("[REFLECTION]:", thought)
    return thought


my_token = ""
model = InferenceClientModel(
    model_id="Qwen/Qwen3-8B", 
    token=my_token,
    tool_choice="auto"
)

# CREATE AGENT 
agent = ToolCallingAgent(
    tools=[sql_engine, reflect_on_observation],
    model=model,
    max_steps=10,
    prompt_templates= my_templates_dict
)


question = "Give the title and the characters name of the most recent work of Shakespeare."
evidence = "characters name refers to CharName; most recent work refers to max(Date)"

full_prompt = f"""
{schema_text}

User Context: {evidence}
Question: {question}
"""

print("Agent is thinking...")
result = agent.run(full_prompt)


Agent is thinking...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ## Database Schema                                                                                              │
│ Table: chapters                                                                                                 │
│   - id (INTEGER)                                                                                                │
│   - Act (INTEGER)                                                                                               │
│   - Scene (INTEGER)                                                                                             │
│   - Description (TEXT)                                                                                          │
│   - work_id (INTEGER)                                                                                           │
│ Table: characters                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - CharName (TEXT)                                                                                             │
│   - Abbrev (TEXT)                                                                                               │
│   - Description (TEXT)                                                                                          │
│ Table: paragraphs                                                                                               │
│   - id (INTEGER)                                                                                                │
│   - ParagraphNum (INTEGER)                                                                                      │
│   - PlainText (TEXT)                                                                                            │
│   - character_id (INTEGER)                                                                                      │
│   - chapter_id (INTEGER)                                                                                        │
│ Table: works                                                                                                    │
│   - id (INTEGER)                                                                                                │
│   - Title (TEXT)                                                                                                │
│   - LongTitle (TEXT)                                                                                            │
│   - Date (INTEGER)                                                                                              │
│   - GenreType (TEXT)                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
│ User Context: characters name refers to CharName; most recent work refers to max(Date)                          │
│ Question: Give the title and the characters name of the most recent work of Shakespeare.                        │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-8B ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'sql_engine' with arguments: {'query': 'SELECT w.Title, c.CharName FROM works w JOIN chapters ch  │
│ ON w.id = ch.work_id JOIN paragraphs p ON ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE │
│ w.id = (SELECT id FROM works ORDER BY Date DESC LIMIT 1);', 'reasoning': 'This query retrieves the title and    │
│ character names from the most recent work by joining works, chapters, paragraphs, and characters tables. The    │
│ subquery finds the most recent work by selecting the maximum Date.'}                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[THOUGHT]: This query retrieves the title and character names from the most recent work by joining works, chapters, paragraphs, and characters tables. The subquery finds the most recent work by selecting the maximum Date.


...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'SELECT w.Title, c.CharName FROM works w JOIN chapters  │
│ ch ON w.id = ch.work_id JOIN paragraphs p ON ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id    │
│ WHERE w.id = (SELECT id FROM works ORDER BY Date DESC LIMIT 1);'}                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON 
ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.id = (SELECT id FROM works ORDER BY Date 
DESC LIMIT 1);

Final answer: SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON 
ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.id = (SELECT id FROM works ORDER BY Date 
DESC LIMIT 1);

[Step 3: Duration 7.10 seconds| Input tokens: 3,224 | Output tokens: 2,462]

In [49]:
gt_query = """SELECT DISTINCT w.Title, ch.CharName FROM works w JOIN chapters c ON w.id = c.work_id JOIN paragraphs p ON p.chapter_id = c.id JOIN characters ch ON ch.id = p.character_id WHERE w.date = ( SELECT max(date) FROM works w2 )"""  
pred_query = """SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON 
ch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.id = (SELECT id FROM works ORDER BY Date 
DESC LIMIT 1);"""

In [50]:
def compute_execution_accuracy(gt_results, predict_results):
  num_correct = 0
  num_queries = len(gt_results)
  mismatch_idx = []

  for i, result in enumerate(gt_results):
      if set(result['results']) == set(predict_results[i]['results']):
          num_correct += 1
      else:
          mismatch_idx.append(i)

  acc = (num_correct / num_queries)

  return acc

import sqlite3
def run_query(db_path, query):
  conn = sqlite3.connect(db_path)
  cursor = conn.cursor()
  cursor.execute(query)
  rows = cursor.fetchall()
  conn.close()

  # Flatten results and convert to list of strings
  return [row[0] for row in rows]

db_name = "shakespeare.sqlite"
rows_gt = run_query(db_name, gt_query)
gt_res = [{"results": rows_gt}]

rows_pred = run_query(db_name, pred_query)
pred_res = [{"results": rows_pred}]

acc = compute_execution_accuracy(gt_res, pred_res)
print(f"Accuracy of the generated SQL query: {acc}")

Accuracy of the generated SQL query: 1.0


In [51]:
complete_trace = {
    "input": full_prompt,
    "output": "",
    "pred_query": pred_query,
    "target_query": gt_query,
    "execution_accuracy": int(acc)
}
complete_trace

{'input': '\n## Database Schema\nTable: chapters\n  - id (INTEGER)\n  - Act (INTEGER)\n  - Scene (INTEGER)\n  - Description (TEXT)\n  - work_id (INTEGER)\nTable: characters\n  - id (INTEGER)\n  - CharName (TEXT)\n  - Abbrev (TEXT)\n  - Description (TEXT)\nTable: paragraphs\n  - id (INTEGER)\n  - ParagraphNum (INTEGER)\n  - PlainText (TEXT)\n  - character_id (INTEGER)\n  - chapter_id (INTEGER)\nTable: works\n  - id (INTEGER)\n  - Title (TEXT)\n  - LongTitle (TEXT)\n  - Date (INTEGER)\n  - GenreType (TEXT)\n\n\nUser Context: characters name refers to CharName; most recent work refers to max(Date)\nQuestion: Give the title and the characters name of the most recent work of Shakespeare.\n',
 'output': '',
 'pred_query': 'SELECT w.Title, c.CharName FROM works w JOIN chapters ch ON w.id = ch.work_id JOIN paragraphs p ON \nch.id = p.chapter_id JOIN characters c ON p.character_id = c.id WHERE w.id = (SELECT id FROM works ORDER BY Date \nDESC LIMIT 1);',
 'target_query': 'SELECT DISTINCT w.Titl